# Pure Pursuit Path Following for a Differential Drive Robot
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Umadin1827/robot-project/blob/main/lecture.ipynb)

**Scientific Programming for Physics · Final assignment (one-hour notebook lecture)**
Author: Umadin1827 · Total time: 60 min

## 0. Title, learning objectives, roadmap (3 min)

In my first notebook my robot drove from waypoint to waypoint. It worked, but it was slow and it did not stay on the path.
Now I improve it with **pure pursuit**, a very popular path tracking method. I also check my code against the pure pursuit example from the PythonRobotics library.

**By the end of this lecture you can:**
1. draw the pure pursuit geometry and write the formula $\kappa = 2\sin\alpha / L_d$;
2. turn a curvature into voltages for the left and right DC motor of a differential drive robot;
3. measure how well a robot follows a path (cross-track error) and compare two controllers;
4. explain what the look-ahead distance $L_d$ does and choose a good value.

| # | Section | min |
|---|---------|-----|
| 0 | Title, objectives, roadmap | 3 |
| 1 | Motivation | 5 |
| 2 | Physics background | 10 |
| 3 | Model setup | 5 |
| 4 | Method and processing | 12 |
| 5 | Verification | 5 |
| 6 | Main result vs source | 5 |
| 7 | Audience exercise | 10 |
| 8 | Summary and questions | 5 |

**Route:** A (reproduction of a published method). Target = the pure pursuit example of PythonRobotics (trajectory of its `main()`), plus my robot version of it.

**Sources**
- R. C. Coulter, *Implementation of the Pure Pursuit Path Tracking Algorithm*, CMU-RI-TR-92-01, Carnegie Mellon University (1992). The original description of the method.
- A. Sakai et al., *PythonRobotics: a Python code collection of robotics algorithms*, arXiv:1808.10703 (2018). Code (MIT licence): [pure_pursuit.py](https://github.com/AtsushiSakai/PythonRobotics/blob/master/PathTracking/pure_pursuit/pure_pursuit.py).
- My own first notebook "Differential drive robot path following": the motor model, the kinematics and the waypoint controller (marked **[my first notebook]** below).

*AI assistant use is declared in the README.*

**How to use it:** press *Runtime → Run all*. The first cell downloads the repository and checks the packages.

In [ ]:
# Setup 1/2: get the repository (needed on a fresh Colab) and install packages that are missing
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError

if not os.path.exists("src"):                      # on a fresh Colab the repo is not there yet
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/Umadin1827/robot-project.git"], check=True)
    os.chdir("robot-project")

missing, other = [], []                            # compare with the versions pinned in requirements.txt
for line in open("requirements.txt"):
    if "==" not in line:
        continue
    name, wanted = line.strip().split("==")
    try:
        have = version(name)
        if have != wanted:
            other.append(f"{name} {have} (pinned {wanted})")
    except PackageNotFoundError:
        missing.append(line.strip())
if missing:                                        # install only what is really missing (no restart needed)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
print("installed now:", missing if missing else "nothing")
print("other versions in this session:", other if other else "none, all pinned versions are used")

In [ ]:
# Setup 2/2: imports, readable plots, one global seed
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display

plt.rcParams.update({"font.size": 14, "axes.grid": True, "figure.figsize": (8, 5),
                     "axes.titlesize": 15})      # big font for the projector
SEED = 0                                          # every random thing uses this seed

from src.paths import spiral_path, straight_path, path_curvature, path_length, wrap_angle
from src.robot import RobotParams, simulate
from src.controllers import (WaypointP, PurePursuit, ConstantVoltage, find_target_index,
                             curvature, wheel_speeds, speed_to_voltage)
from src.motor import dc_motor, motor_step_exact, motor_step_ivp, K_E, T_M
from src.metrics import cross_track_error, summary
from src.plotting import plot_run
from src.bicycle import CarParams, simulate_car, source_course
print("ready")

## 1. Motivation (5 min)

My robot has two wheels with a motor each (differential drive). I want it to follow a path, here a spiral with two loops.

In my first notebook the controller was very simple: *"look at the next waypoint, turn towards it, drive to it, then take the next one."*
The robot has to get within 5 cm of every waypoint before it goes on.

> 🔮 **Predict before you run:** will the blue line stay on the red dashed path? Where do you expect the biggest problems?

In [ ]:
p = RobotParams()                                  # robot numbers (explained in section 3)
wx, wy = spiral_path(21)                           # 21 waypoints, as in my first notebook
cx, cy = spiral_path(800)                          # same spiral as a dense path (used later)
old = simulate(WaypointP(wx, wy), p)               # my first controller [my first notebook]

fig, ax = plt.subplots(figsize=(11, 4.5))
plot_run(ax, cx, cy, old, "waypoint P controller")
ax.plot(wx, wy, "go", ms=7, label="waypoints")
ax.set_title("My first controller: waypoint to waypoint"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=4, fontsize=11)
plt.show()
cte_old = cross_track_error(old.x, old.y, cx, cy)
print(f"time needed: {old.t[-1]:.0f} s   |   largest distance from the path: {cte_old.max()*100:.1f} cm")

It reaches all waypoints and it is not too bad, but look at the numbers: it needs a long time and it is quite far from the path at some moments. Two reasons. The voltage is proportional to the distance to the next waypoint, so the robot slows down every time it comes close to one. And it only knows the *next* point, not the shape of the path, so it reacts late. The biggest error is at the very beginning: the robot starts looking along $x$, but the path starts going up.

**Question for today:** can the robot look a bit *ahead* on the path and follow it smoothly, without stopping?

## 2. Physics background (10 min)

### 2.1 Kinematics of the differential drive robot
Two wheels, distance $B$ between them, wheel radius $r$. If the wheels have linear speeds $v_l$ and $v_r$:

$$v = \frac{v_r + v_l}{2}, \qquad \omega = \frac{v_r - v_l}{B}$$

$$\dot x = v\cos\theta, \qquad \dot y = v\sin\theta, \qquad \dot\theta = \omega$$

| symbol | meaning | unit |
|---|---|---|
| $v$ | speed of the robot centre | m/s |
| $\omega$ | turning rate | rad/s |
| $\theta$ | heading (course) angle | rad |
| $B$, $r$ | wheel distance, wheel radius | m |
| $v_{l,r} = r\,\omega_{l,r}$ | wheel speed from motor angular speed | m/s |

The robot drives on a circle of radius $R = v/\omega$. The **curvature** is $\kappa = 1/R = \omega / v$ (unit 1/m, positive = left turn).

### 2.2 DC motor
Each wheel has a DC motor. With voltage $u$ the speed $\omega_m$ follows (no viscous friction, as in my first notebook):

$$T_m\,\dot\omega_m + \omega_m = \frac{u}{k_e}$$

After a time of a few $T_m$ the speed is constant: $\omega_{ss} = u/k_e$. So the motor is a "first order lag": it reaches the wanted speed, but not instantly.

### 2.3 The pure pursuit idea (Coulter 1992)
1. Look at the path **ahead of the robot** and pick the point that is the distance $L_d$ away (the *look-ahead distance*). This is the target point.
2. Find the circle that goes through the robot and the target point and touches the robot heading.
3. Drive on this circle. Then repeat everything at the next time step.

**The formula.** Let $\alpha$ be the angle between the robot heading and the line to the target. The line to the target is a chord of the circle with length $L_d$. A chord and the tangent at its end make the angle $\alpha$, and the chord length is $2R\sin\alpha$. So

$$L_d = 2R\sin\alpha \quad\Rightarrow\quad \kappa = \frac{1}{R} = \frac{2\sin\alpha}{L_d}$$

With the speed $v$ chosen by us, the wheels get $v_l = v\,(1-\kappa B/2)$ and $v_r = v\,(1+\kappa B/2)$, and the motor voltage is $u = k_e\,v_{wheel}/r$.

In [ ]:
# the picture of the pure pursuit geometry (a made-up example, only to explain)
Ld_demo, alpha_demo = 1.0, np.radians(30)
tx, ty = Ld_demo*np.cos(alpha_demo), Ld_demo*np.sin(alpha_demo)       # target point
kappa_demo = 2*np.sin(alpha_demo)/Ld_demo
R_demo = 1/kappa_demo
phi = np.linspace(0, 2*np.pi, 200)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(R_demo*np.sin(phi), R_demo*(1-np.cos(phi)), "g-", label="circle to drive on")  # circle touches x axis at robot
ax.annotate("", xy=(0.7, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", lw=3))
ax.plot([0, tx], [0, ty], "k:", label="$L_d$")
ax.plot(0, 0, "bs", ms=12, label="robot"); ax.plot(tx, ty, "r*", ms=18, label="target point")
ax.text(0.45, 0.05, "heading", fontsize=13); ax.text(0.35, 0.12, r"$\alpha$", fontsize=16)
ax.set(xlabel="x [m]", ylabel="y [m]", title="Pure pursuit geometry", aspect="equal"); ax.legend(loc="upper left", fontsize=12)
plt.show()

### 2.4 A small analytic example
Robot at the origin, heading along $x$. Target at the distance $L_d = 1$ m with $\alpha = 30°$.

> 🔮 **Predict:** what is $\kappa$? And what is the radius $R$ of the circle? (Hint: $\sin 30° = 0.5$.) What if $\alpha = 0$?

In [ ]:
print(f"alpha = 30 deg, Ld = 1 m   : kappa = {curvature(np.radians(30), 1.0):.3f} 1/m  (R = {1/curvature(np.radians(30), 1.0):.2f} m)")
print(f"alpha =  0 deg, Ld = 1 m   : kappa = {curvature(0.0, 1.0):.3f} 1/m  -> straight line")
print(f"alpha = 90 deg, Ld = 0.5 m : R = {1/curvature(np.radians(90), 0.5):.2f} m  -> very sharp turn")

So the robot turns more when the target is far to the side (big $\alpha$) and when $L_d$ is small. This gives us the main design knob: **$L_d$**.

## 3. Model setup (5 min)

All robot numbers are in one `dataclass` (`RobotParams`). The values come from **my first notebook** (my own robot model), they are not guessed and not taken from the PythonRobotics paper. The motor constants `T_M` and `K_E` live in `src/motor.py`.
The step `dt` is 0.01 s, the voltage limit is ±8 V (like the PWM limit in the first notebook).

In [ ]:
print(p)                                           # Technique: dataclass prints all fields nicely
print(f"motor: T_M = {T_M} s,  K_E = {K_E} V*s/rad")
print(f"top speed of one wheel at {p.u_max} V: v_max = {p.v_max:.3f} m/s")

**Python techniques in this notebook, and why I use them**

| technique | where | why |
|---|---|---|
| `dataclass` | `RobotParams`, `Result` | all parameters/results in one object instead of 20 loose variables |
| `scipy.integrate.solve_ivp` | `motor_step_ivp` | ready ODE solver, I use it as the trusted reference for the motor |
| `scipy.spatial.cKDTree` | `cross_track_error` | fast "closest path point" search for 10 000 robot positions, without a Python loop |
| `typing.Protocol` | `Controller` | `simulate()` accepts any controller that has `command()` and `finished` |
| seeded `np.random.default_rng` | `simulate(noise_std=...)` | noisy sensor, but the same result every run |
| `matplotlib.animation.FuncAnimation` | section 4.5 | movie of the robot |
| `assert` / `np.testing` + `pytest` | sections 5 and 6 | automatic checks |

Now the path. The dense path has 800 points. Pure pursuit needs many close points, the waypoint controller only needs 21.

In [ ]:
kappa_path = path_curvature(cx, cy)                # 1/R of the path at every point
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
axs[0].plot(cx, cy, "r-", label="dense path (800 points)"); axs[0].plot(wx, wy, "go", label="21 waypoints")
axs[0].set(xlabel="x [m]", ylabel="y [m]", title="The path", aspect="equal"); axs[0].legend(fontsize=11)
axs[1].plot(np.linspace(0, path_length(cx, cy), len(cx)), kappa_path)
axs[1].set(xlabel="distance along path [m]", ylabel="curvature [1/m]", title="Curvature of the path")
plt.tight_layout(); plt.show()
R_min = 1/np.abs(kappa_path).max()
print(f"path length: {path_length(cx, cy):.2f} m,  tightest turn: R = {R_min:.2f} m  =  {R_min/p.B:.1f} x the wheel distance B")

The tightest turn of the spiral is not much bigger than the robot itself (see the ratio $R_{min}/B$ above). So the path is demanding, and we must choose $L_d$ small enough.

## 4. Method and processing (12 min)
I build the controller in five small steps. After every step there is a plot or a check.

### 4.1 The motor: voltage → wheel speed  [my first notebook]
`dc_motor` is the ODE of the motor from my first notebook. `solve_ivp` solves it numerically. For the 40000 steps of the simulation this was slow, so in `src/motor.py` I also wrote the **exact solution** of this linear ODE for a constant voltage (`motor_step_exact`).

> 🔮 **Predict:** the voltage is 8 V all the time. What will the motor speed be after 1 s? (Use $\omega_{ss} = u/k_e$.)

In [ ]:
dt, t_end = 0.01, 1.0
state_ivp, state_ex = [0.0, 0.0], (0.0, 0.0)       # [angle, speed] of the motor
t_list, w_ivp, w_ex = [0.0], [0.0], [0.0]
for k in range(int(t_end/dt)):
    state_ivp = motor_step_ivp(state_ivp, 8.0, k*dt, (k+1)*dt)   # solve_ivp every step
    state_ex = motor_step_exact(state_ex, 8.0, dt)               # exact formula
    t_list.append((k+1)*dt); w_ivp.append(state_ivp[1]); w_ex.append(state_ex[1])

plt.plot(t_list, w_ex, lw=3, label="exact formula"); plt.plot(t_list[::5], w_ivp[::5], "ro", label="solve_ivp")
plt.axhline(8.0/K_E, color="gray", ls="--", label="u / k_e")
plt.xlabel("time [s]"); plt.ylabel("motor speed [rad/s]"); plt.title("Motor step response, u = 8 V"); plt.legend(); plt.show()
print(f"speed after 1 s: {w_ex[-1]:.2f} rad/s  ->  wheel speed {w_ex[-1]*p.r:.3f} m/s")

The two methods lie on the same curve, and the motor needs a few $T_m$ to be fast. (The exact check with numbers is in section 5.)

### 4.2 Kinematics: from wheel angles to robot position  [my first notebook]
From the two motor angles I get the distance travelled and the heading (`update_pose` in `src/robot.py`):

`path = (theta_r + theta_l) * r / 2`, `course = (theta_r - theta_l) * r / B`, and the robot moves by `path` in the direction `course`.

> 🔮 **Predict:** left motor 4 V, right motor 6 V, all the time. Which way does the robot turn, and what does the track look like?

In [ ]:
res_circle = simulate(ConstantVoltage(4.0, 6.0), p, t_max=15)    # same voltages all the time
plt.plot(res_circle.x, res_circle.y, lw=3); plt.plot(0, 0, "ks", ms=10)
plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.title("Constant voltages: 4 V left, 6 V right")
plt.gca().set_aspect("equal"); plt.show()

The right wheel is faster, so the robot turns **left** and drives on a circle. The first part is not a perfect circle because the motors need time to speed up.

### 4.3 Find the target point
`find_target_index` does two things: (1) from the last nearest point it goes forward while the next point is closer (it never goes back, so the loops of the spiral cannot confuse it); (2) from there it walks forward along the path until a point is at least $L_d$ away. That is the target.
Here the robot is placed a bit beside the path and a bit turned away.

In [ ]:
i0 = 200                                           # a point on the path
yaw0 = np.arctan2(cy[i0+1]-cy[i0], cx[i0+1]-cx[i0]) - 0.4     # path direction, turned by -0.4 rad
x0, y0 = cx[i0] + 0.05, cy[i0] - 0.10             # a bit beside the path
Ld = 0.3
near, target = find_target_index(cx, cy, x0, y0, nearest=i0-20, Ld=Ld)   # search starts 20 points behind

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cx[120:320], cy[120:320], "r--", lw=2, label="path")
ax.plot(x0 + Ld*np.cos(phi), y0 + Ld*np.sin(phi), "gray", ls=":", label="distance $L_d$")
ax.annotate("", xy=(x0+0.15*np.cos(yaw0), y0+0.15*np.sin(yaw0)), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", lw=3))
ax.plot(x0, y0, "bs", ms=12, label="robot"); ax.plot(cx[near], cy[near], "go", ms=10, label="nearest path point")
ax.plot(cx[target], cy[target], "r*", ms=20, label="target point")
ax.set(xlabel="x [m]", ylabel="y [m]", title="Target point search", aspect="equal"); ax.legend(fontsize=11); plt.show()
print("nearest index:", near, "  target index:", target, "  distance to target:", np.hypot(cx[target]-x0, cy[target]-y0).round(3), "m")

### 4.4 From target to wheel voltages
Now the three formulas: angle $\alpha$ → curvature $\kappa$ → wheel speeds → voltages. The chosen speed is `v = 0.25 m/s`, which is below the top speed of the wheel, so there is room to turn.

In [ ]:
v = 0.25                                           # chosen robot speed [m/s]
alpha = wrap_angle(np.arctan2(cy[target]-y0, cx[target]-x0) - yaw0)   # angle between heading and target
kappa = curvature(alpha, Ld)                       # pure pursuit formula
v_l, v_r = wheel_speeds(v, kappa, p.B)             # wheel speeds [m/s]
u_l, u_r = speed_to_voltage(v_l, p), speed_to_voltage(v_r, p)         # motor voltages [V]
print(f"alpha = {np.degrees(alpha):.1f} deg,  kappa = {kappa:.2f} 1/m  (R = {1/abs(kappa):.2f} m)")
print(f"v_left = {v_l:.3f} m/s,  v_right = {v_r:.3f} m/s   ->   u_left = {u_l:.2f} V,  u_right = {u_r:.2f} V")
print(f"check: (v_r - v_l)/B = {(v_r-v_l)/p.B:.3f} rad/s   and   v*kappa = {v*kappa:.3f} rad/s")

In [ ]:
# the circle the robot will drive on = circle through robot and target that touches the heading
cxc, cyc = x0 - np.sin(yaw0)/kappa, y0 + np.cos(yaw0)/kappa           # centre of the circle
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(cx[120:320], cy[120:320], "r--", lw=2, label="path")
ax.plot(cxc + abs(1/kappa)*np.cos(phi), cyc + abs(1/kappa)*np.sin(phi), "g-", label="circle from the formula")
ax.plot(x0, y0, "bs", ms=12, label="robot"); ax.plot(cx[target], cy[target], "r*", ms=20, label="target")
ax.set(xlim=(x0-0.6, x0+0.6), ylim=(y0-0.5, y0+0.7), xlabel="x [m]", ylabel="y [m]", title="The circle goes through robot and target")
ax.set_aspect("equal"); ax.legend(fontsize=11); plt.show()

The green circle really goes through the target. (The check in section 5 shows this exactly for a circle path.)

### 4.5 Put it all in a loop
`simulate()` repeats at every 0.01 s step: measure pose → controller gives two voltages → motors move → new pose. The controller is my class `PurePursuit`; it finishes when the robot is within 5 cm of the last path point.

In [ ]:
Ld, v_ref = 0.3, 0.25                              # look-ahead [m] and speed [m/s]
new = simulate(PurePursuit(cx, cy, p, Ld=Ld, v_ref=v_ref), p)

fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
plot_run(axs[0], cx, cy, new, "pure pursuit"); axs[0].set_title("Pure pursuit on the spiral")
axs[1].plot(new.t, new.u_l, label="left motor"); axs[1].plot(new.t, new.u_r, label="right motor")
axs[1].set(xlabel="time [s]", ylabel="motor voltage [V]", title="Control signals"); axs[1].legend(fontsize=11)
plt.tight_layout(); plt.show()
print("finished:", new.finished, f"  time: {new.t[-1]:.1f} s")

In [ ]:
# Technique: matplotlib FuncAnimation - a movie of the robot (set SHOW_ANIMATION = False to skip it)
from matplotlib import animation
SHOW_ANIMATION = True
if SHOW_ANIMATION:
    step = max(1, len(new.x) // 90)                # about 90 frames
    xs, ys = new.x[::step], new.y[::step]
    fig, ax = plt.subplots(figsize=(10, 4.5), dpi=70)
    ax.plot(cx, cy, "r--", alpha=0.5, label="wanted path")
    track, = ax.plot([], [], "b-", lw=2, label="robot track"); body, = ax.plot([], [], "ko", ms=10)
    ax.set(xlabel="x [m]", ylabel="y [m]", title="Pure pursuit, animation", aspect="equal"); ax.legend(loc="upper left", fontsize=10)
    def update(f):                                 # draws frame number f
        track.set_data(xs[:f+1], ys[:f+1]); body.set_data([xs[f]], [ys[f]]); return track, body
    anim = animation.FuncAnimation(fig, update, frames=len(xs), interval=80, blit=True)
    plt.close(fig); display(HTML(anim.to_jshtml()))

## 5. Verification (5 min)
Does the code do what the physics says? Four checks against a known answer. If one is wrong, the cell stops with an error (`assert`). The same checks are also in `tests/` (pytest).

**Check 1: the fast exact motor formula = the numerical solver.**

In [ ]:
exact = motor_step_exact((0.3, 2.0), u=5.0, dt=0.01)
ivp = motor_step_ivp([0.3, 2.0], 5.0, 0.0, 0.01)
np.testing.assert_allclose(exact, ivp, rtol=1e-5)
print("check 1 OK  exact:", np.round(exact, 6), " solve_ivp:", np.round(ivp, 6))

**Check 2: analytic limit.** A robot on a circle of radius $R$, heading along the tangent. A point on the circle at chord length $L_d$ has $\alpha = \arcsin(L_d/2R)$. The formula must give back exactly $1/R$.

In [ ]:
for R in (0.5, 1.0, 3.0):
    Ld_t = 0.4
    alpha_t = np.arcsin(Ld_t / (2*R))                       # exact angle for a circle of radius R
    np.testing.assert_allclose(curvature(alpha_t, Ld_t), 1/R, rtol=1e-12)
    print(f"check 2 OK  R = {R} m:  kappa = {curvature(alpha_t, Ld_t):.6f}  1/R = {1/R:.6f}")

**Check 3: constant voltages give a circle with the radius from the kinematics**, $R = \frac{B}{2}\frac{v_r+v_l}{v_r-v_l}$. I measure the radius in the simulation as (distance driven) / (heading change) at the end, when the motors are already steady.

In [ ]:
res3 = simulate(ConstantVoltage(4.0, 6.0), p, t_max=20)
v_l3, v_r3 = 4.0/K_E*p.r, 6.0/K_E*p.r
R_expected = p.B/2 * (v_r3 + v_l3)/(v_r3 - v_l3)
d_yaw = res3.yaw[-1] - res3.yaw[-500]
dist = np.sum(np.hypot(np.diff(res3.x[-500:]), np.diff(res3.y[-500:])))
np.testing.assert_allclose(dist/d_yaw, R_expected, rtol=1e-3)
print(f"check 3 OK  R from simulation = {dist/d_yaw:.4f} m,  R from formula = {R_expected:.4f} m")

**Check 4: convergence.** Start 0.5 m beside a straight line. Pure pursuit must bring the robot onto the line.

In [ ]:
sx, sy = straight_path(6, 600)
res4 = simulate(PurePursuit(sx, sy, p, Ld=0.3), p, t_max=100, pose0=(0.0, 0.5, 0.0))   # starts at y = 0.5 m
assert res4.finished and abs(res4.y[-1]) < 0.01
plt.plot(res4.x, res4.y, lw=3, label="robot"); plt.plot(sx, sy, "r--", label="line")
plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.title("Check 4: robot converges to the line"); plt.legend(); plt.show()
print(f"check 4 OK  final distance to the line: {abs(res4.y[-1])*1000:.2f} mm")

In [ ]:
!python -m pytest -q --color=no tests

## 6. Main result vs source (5 min)

### 6.1 Does my pure pursuit give the same result as PythonRobotics?
The source simulates a **car** (bicycle model) with pure pursuit. I re-wrote its algorithm in `src/bicycle.py` (same parameters, same course: $y = \sin(x/5)\,x/2$) and compare with the trajectory of the **original code**. That trajectory is saved in `data/pythonrobotics_reference.csv` by `scripts/make_reference.py`, which downloads the original file from GitHub and runs it.

My success criterion: the position difference stays below $10^{-9}$ m. (Both codes do the same operations in the same order, so I expect almost zero.)

In [ ]:
ref = np.loadtxt("data/pythonrobotics_reference.csv", delimiter=",", skiprows=1)   # columns: t, x, y, yaw, v, steer
mine = simulate_car(*source_course())
diff = np.hypot(ref[:, 1] - mine[:, 1], ref[:, 2] - mine[:, 2])
assert ref.shape == mine.shape and diff.max() < 1e-9

course_x, course_y = source_course()
fig, axs = plt.subplots(1, 3, figsize=(19, 4.8))
for ax, data, name in [(axs[0], ref, "original PythonRobotics code"), (axs[1], mine, "my re-implementation")]:
    ax.plot(course_x, course_y, "r.", ms=4, label="course"); ax.plot(data[:, 1], data[:, 2], "b-", label="car")
    ax.set(xlabel="x [m]", ylabel="y [m]", title=name); ax.legend(fontsize=11)
axs[2].semilogy(mine[:, 0], np.maximum(diff, 1e-16), label="difference"); axs[2].axhline(1e-9, color="r", ls="--", label="criterion")
axs[2].set(xlabel="time [s]", ylabel="position difference [m]", title="Residual (mine - original)"); axs[2].legend(fontsize=11)
plt.tight_layout(); plt.show()
print(f"largest position difference: {diff.max():.1e} m  (criterion 1e-9 m)   ->  OK")
print(f"largest steering angle difference: {np.abs(ref[:,5]-mine[:,5]).max():.1e} rad")

No discrepancy above my criterion. Two honest remarks: (1) I wrote my version while reading the source, so this shows that I copied the algorithm *correctly*; the check that the formula itself is right is section 5. (2) The source needs `angle_mod` from its own `utils` folder; I used the same formula $(a+\pi) \bmod 2\pi - \pi$.

### 6.2 My robot: pure pursuit vs my first controller
Same spiral, same robot, same measure: the **cross-track error** = distance from the robot to the closest point of the path.

In [ ]:
cte_new = cross_track_error(new.x, new.y, cx, cy)  # Technique: cKDTree nearest-neighbour search inside this function
s_old, s_new = summary(cte_old, old.t), summary(cte_new, new.t)
print(f"{'controller':32s}{'RMS error [cm]':>16s}{'max error [cm]':>16s}{'time [s]':>10s}")
for name, s in [("waypoint P (first notebook)", s_old), (f"pure pursuit (Ld = {Ld} m)", s_new)]:
    print(f"{name:32s}{s['rms_cte']*100:16.2f}{s['max_cte']*100:16.2f}{s['time']:10.1f}")
print(f"largest error of pure pursuit at t = {new.t[np.argmax(cte_new)]:.1f} s (the start), waypoint P at t = {old.t[np.argmax(cte_old)]:.1f} s")
print(f"-> RMS error is {100*(1 - s_new['rms_cte']/s_old['rms_cte']):.0f} % smaller, the robot is {s_old['time']/s_new['time']:.1f} times faster")

fig, axs = plt.subplots(1, 2, figsize=(15, 4.8))
axs[0].plot(cx, cy, "r--", lw=2, label="path"); axs[0].plot(old.x, old.y, label="waypoint P"); axs[0].plot(new.x, new.y, label="pure pursuit")
axs[0].set(xlabel="x [m]", ylabel="y [m]", title="Both controllers", aspect="equal"); axs[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=3, fontsize=11)
axs[1].plot(old.t, cte_old*100, label="waypoint P"); axs[1].plot(new.t, cte_new*100, label="pure pursuit")
axs[1].set(xlabel="time [s]", ylabel="distance from path [cm]", title="Cross-track error"); axs[1].legend(fontsize=11)
plt.tight_layout(); plt.show()

### 6.3 How big should $L_d$ be?
I run the robot with different look-ahead distances (noise-free sensor).

In [ ]:
ld_values = [0.05, 0.1, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0]
rms_list, time_list = [], []
for Ld_try in ld_values:
    r_try = simulate(PurePursuit(cx, cy, p, Ld=Ld_try, v_ref=0.25), p)
    rms_list.append(summary(cross_track_error(r_try.x, r_try.y, cx, cy), r_try.t)["rms_cte"] * 100)
    time_list.append(r_try.t[-1])

fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
axs[0].plot(ld_values, rms_list, "o-"); axs[0].set(xlabel="look-ahead distance $L_d$ [m]", ylabel="RMS error [cm]", title="Bigger $L_d$: robot cuts corners")
axs[1].plot(ld_values, time_list, "o-", color="tab:green"); axs[1].set(xlabel="look-ahead distance $L_d$ [m]", ylabel="time to finish [s]", title="Time to finish")
plt.tight_layout(); plt.show()

## 7. Audience exercise (10 min)
Three tasks. Use the cells below and change them. (Solutions: `solutions.ipynb` in the repository, please do not open it before you tried.)

### 🟢 Exercise 1 (easy): change a parameter
Change `Ld_ex` to **0.8**, run the cell and describe in one sentence what happens to the robot track. Then try **0.05**: is the error better or worse, and what do you see on the control signals? (Is a small $L_d$ always the best? Exercise 3 will tell.)

In [ ]:
Ld_ex = 0.3                                        # <- change me (try 0.8 and 0.05)
res_ex = simulate(PurePursuit(cx, cy, p, Ld=Ld_ex, v_ref=0.25), p, t_max=200)
fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
plot_run(axs[0], cx, cy, res_ex, f"Ld = {Ld_ex} m"); axs[0].set_title("Track")
axs[1].plot(res_ex.t, res_ex.u_l, label="left"); axs[1].plot(res_ex.t, res_ex.u_r, label="right")
axs[1].set(xlabel="time [s]", ylabel="voltage [V]", title="Control signals"); axs[1].legend()
plt.tight_layout(); plt.show()
cte_ex = cross_track_error(res_ex.x, res_ex.y, cx, cy)
print(f"RMS error {np.sqrt(np.mean(cte_ex**2))*100:.2f} cm,  time {res_ex.t[-1]:.1f} s,  finished: {res_ex.finished}")

### 🟡 Exercise 2 (medium): complete a function
Complete `my_wheel_speeds` so that the robot drives with speed `v` on a circle with curvature `kappa`.
Hint: $\omega = v\kappa$ and $v_r - v_l = B\,\omega$, and the mean of the two wheel speeds is $v$.

In [ ]:
def my_wheel_speeds(v, kappa, B):
    """Return (v_left, v_right) in m/s."""
    # TODO: replace the two lines below with the correct formulas
    v_left = v
    v_right = v
    return v_left, v_right

# automatic check (does not stop the notebook, it only tells you)
ok = all(np.allclose(my_wheel_speeds(0.2, k, 0.15), wheel_speeds(0.2, k, 0.15)) for k in (0.0, 1.0, -2.0))
print("✅ correct!" if ok else "❌ not yet, look at the hint")

### 🔴 Exercise 3 (stretch): a noisy position sensor
A real robot does not know its position exactly. `simulate(..., noise_std=0.1)` adds Gaussian noise (σ = 10 cm, seeded) to the measured position.
The starter cell sweeps $L_d$ and plots the RMS error. **TODO:** also compute the *chatter* of the left motor voltage, `np.std(np.diff(res.u_l))`, plot it against $L_d$ in a second panel, and decide which $L_d$ you would use on the real robot and why.

In [ ]:
ld_noisy = [0.1, 0.2, 0.3, 0.5, 0.8]
rms_noisy = []
for Ld_try in ld_noisy:
    r_try = simulate(PurePursuit(cx, cy, p, Ld=Ld_try, v_ref=0.25), p, t_max=200, noise_std=0.1, seed=SEED)
    rms_noisy.append(np.sqrt(np.mean(cross_track_error(r_try.x, r_try.y, cx, cy)**2)) * 100)
    # TODO: store np.std(np.diff(r_try.u_l)) in a second list here
plt.plot(ld_noisy, rms_noisy, "o-"); plt.xlabel("look-ahead distance $L_d$ [m]"); plt.ylabel("RMS error [cm]")
plt.title("Noisy sensor, sigma = 10 cm"); plt.show()

## 8. Summary and questions (5 min)

**Take-aways**
1. Pure pursuit is one formula, $\kappa = 2\sin\alpha/L_d$: look ahead, aim at the target, drive on the circle.
2. For a differential drive robot the curvature gives two wheel speeds, and the motor law $u = k_e v/r$ gives the voltages.
3. It has two knobs: the speed $v$ and the look-ahead $L_d$. Large $L_d$ is smooth but cuts corners (section 6.3). Small $L_d$ follows the path closely, but it reacts strongly to every error, also to sensor noise (exercise 3).
4. My implementation matches the PythonRobotics example, passes four physics checks, and on my spiral it is better than my first waypoint controller.

In [ ]:
sat = np.mean((np.abs(new.u_l) >= p.u_max - 1e-9) | (np.abs(new.u_r) >= p.u_max - 1e-9)) * 100
print(f"Numbers of this lecture (all computed above):")
print(f"  RMS error: {s_old['rms_cte']*100:.2f} cm (waypoint P)  ->  {s_new['rms_cte']*100:.2f} cm (pure pursuit)")
print(f"  time:      {s_old['time']:.0f} s  ->  {s_new['time']:.0f} s")
print(f"  voltage at the limit during {sat:.1f} % of the pure pursuit run")
print(f"  source reproduction: largest position difference {diff.max():.1e} m")

**Limitations**
- It is only a simulation: no wheel slip, no sensor delay, perfect knowledge of the heading.
- The speed is constant. A real robot should slow down in sharp turns (the tightest turn here is already demanding).
- Pure pursuit needs a dense, smooth path. With a big $L_d$ it cuts corners, with a small $L_d$ it is nervous.
- I did not tune $L_d$ and $v$ together or test on the real hardware.

**Further reading**
- Coulter (1992), CMU-RI-TR-92-01, the original report.
- Sakai et al. (2018), *PythonRobotics*, arXiv:1808.10703, and the code on GitHub.
- Snider (2009), *Automatic Steering Methods for Autonomous Automobile Path Tracking*, CMU-RI-TR-09-08: compares pure pursuit with other path trackers.

**Questions?**